# Discovery + Silver: `crm.contacts`

Personas dentro de una `account`. FK obligatoria a `accounts`.

In [1]:
import sys
sys.path.append("/home/jovyan/work/src")

import pandas as pd
from utils.db import get_engine

engine = get_engine()
df = pd.read_sql("SELECT * FROM bronze.crm__contacts", engine)
df.shape

(15000, 11)

## 1. Forma general

In [2]:
print(df.dtypes)
df.head()

contact_id              object
first_name              object
last_name               object
email                   object
phone                   object
title                   object
created_at              object
account_id              object
_source_file            object
_ingested_at    datetime64[ns]
_dag_run_id             object
dtype: object


,contact_id,first_name,last_name,email,phone,title,created_at,account_id,_source_file,_ingested_at,_dag_run_id
0,CON-0000001,Agustina,Reyes,agustina.reyes5416@synthetic.dev,+56 2 6325 8990,Director,2019-11-14 15:32:18,ACC-0003994,crm/contacts.csv,2026-07-17 15:16:32.777171,manual__2026-07-17T15:16:30+00:00
1,CON-0000002,Amanda,Araya,amanda.araya4596@mail.test,+56 3 4585 4539,VP,2023-04-20 11:58:45,ACC-0002188,crm/contacts.csv,2026-07-17 15:16:32.777171,manual__2026-07-17T15:16:30+00:00
2,CON-0000003,Constanza,Pino,constanza.pino9730@lake.local,+56 3 9644 4573,Sales Rep,2025-12-09 22:55:19,ACC-0001589,crm/contacts.csv,2026-07-17 15:16:32.777171,manual__2026-07-17T15:16:30+00:00
3,CON-0000004,Tomas,Espinoza,tomas.espinoza3400@demo.io,+56 7 5955 9961,CEO,2025-12-28 06:51:07,ACC-0001995,crm/contacts.csv,2026-07-17 15:16:32.777171,manual__2026-07-17T15:16:30+00:00
4,CON-0000005,Camila,Ortiz,camila.ortiz2378@lake.local,+56 5 1366 6556,Manager,2019-10-04 09:46:14,ACC-0001288,crm/contacts.csv,2026-07-17 15:16:32.777171,manual__2026-07-17T15:16:30+00:00


## 2. Nulos, duplicados e integridad referencial

In [3]:
print("Nulos por columna:")
print(df.isna().sum())
print()
print("contact_id duplicados:", df["contact_id"].duplicated().sum())

accounts = pd.read_sql("SELECT account_id FROM silver.crm__accounts", engine)
print("account_id huerfanos:", (~df["account_id"].isin(accounts["account_id"])).sum())

Nulos por columna:
contact_id      0
first_name      0
last_name       0
email           0
phone           0
title           0
created_at      0
account_id      0
_source_file    0
_ingested_at    0
_dag_run_id     0
dtype: int64

contact_id duplicados: 0
account_id huerfanos: 0


## 3. `title`: valores

In [4]:
print("title:")
print(df["title"].value_counts())

title:
title
Analyst       1537
Engineer      1529
CFO           1519
Manager       1514
VP            1502
Specialist    1500
CEO           1491
CTO           1487
Sales Rep     1469
Director      1452
Name: count, dtype: int64


## 4. Conclusion

Tabla limpia (sin nulos, sin duplicados, 0 FKs huerfanas). Solo tipado y estandarizacion.

## 5. Limpieza con pandas

In [5]:
df_silver = df[["contact_id", "account_id", "first_name", "last_name", "email", "phone", "title", "created_at"]].copy()

df_silver["first_name"] = df_silver["first_name"].str.strip()
df_silver["last_name"] = df_silver["last_name"].str.strip()
df_silver["email"] = df_silver["email"].str.strip().str.lower()
df_silver["phone"] = df_silver["phone"].str.strip()
df_silver["title"] = df_silver["title"].str.strip()
df_silver["created_at"] = pd.to_datetime(df_silver["created_at"])

df_silver.head()

,contact_id,account_id,first_name,last_name,email,phone,title,created_at
0,CON-0000001,ACC-0003994,Agustina,Reyes,agustina.reyes5416@synthetic.dev,+56 2 6325 8990,Director,2019-11-14 15:32:18
1,CON-0000002,ACC-0002188,Amanda,Araya,amanda.araya4596@mail.test,+56 3 4585 4539,VP,2023-04-20 11:58:45
2,CON-0000003,ACC-0001589,Constanza,Pino,constanza.pino9730@lake.local,+56 3 9644 4573,Sales Rep,2025-12-09 22:55:19
3,CON-0000004,ACC-0001995,Tomas,Espinoza,tomas.espinoza3400@demo.io,+56 7 5955 9961,CEO,2025-12-28 06:51:07
4,CON-0000005,ACC-0001288,Camila,Ortiz,camila.ortiz2378@lake.local,+56 5 1366 6556,Manager,2019-10-04 09:46:14


## 6. Validar antes de escribir

In [6]:
assert len(df_silver) == len(df)
assert df_silver["contact_id"].is_unique
assert df_silver["account_id"].isin(accounts["account_id"]).all()
print("OK:", len(df_silver), "filas listas para silver")

OK: 15000 filas listas para silver


## 7. Escribir en `silver.crm__contacts`

In [7]:
df_silver["_silver_loaded_at"] = pd.Timestamp.utcnow()

df_silver.to_sql(
    "crm__contacts",
    engine,
    schema="silver",
    if_exists="replace",
    index=False,
    method="multi",
    chunksize=3000,
)
print("Escrito en silver.crm__contacts")

Escrito en silver.crm__contacts


## 8. Verificar

In [8]:
check = pd.read_sql("SELECT * FROM silver.crm__contacts LIMIT 5", engine)
print(pd.read_sql("SELECT count(*) AS filas, count(DISTINCT contact_id) AS ids_unicos FROM silver.crm__contacts", engine))
check

   filas  ids_unicos
0  15000       15000


,contact_id,account_id,first_name,last_name,email,phone,title,created_at,_silver_loaded_at
0,CON-0000001,ACC-0003994,Agustina,Reyes,agustina.reyes5416@synthetic.dev,+56 2 6325 8990,Director,2019-11-14 15:32:18,2026-07-17 15:17:00.148049+00:00
1,CON-0000002,ACC-0002188,Amanda,Araya,amanda.araya4596@mail.test,+56 3 4585 4539,VP,2023-04-20 11:58:45,2026-07-17 15:17:00.148049+00:00
2,CON-0000003,ACC-0001589,Constanza,Pino,constanza.pino9730@lake.local,+56 3 9644 4573,Sales Rep,2025-12-09 22:55:19,2026-07-17 15:17:00.148049+00:00
3,CON-0000004,ACC-0001995,Tomas,Espinoza,tomas.espinoza3400@demo.io,+56 7 5955 9961,CEO,2025-12-28 06:51:07,2026-07-17 15:17:00.148049+00:00
4,CON-0000005,ACC-0001288,Camila,Ortiz,camila.ortiz2378@lake.local,+56 5 1366 6556,Manager,2019-10-04 09:46:14,2026-07-17 15:17:00.148049+00:00
